In [55]:
# 필요한 라이브러리 설치 (최초 한 번만)
# !pip install openai langchain tiktoken

import os
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from dotenv import load_dotenv
load_dotenv()


True

In [60]:
prompt_template = PromptTemplate.from_template("""
너는 인공지능 면접관이다.  
다음은 면접 질문과 지원자의 답변이다.  
아래의 절차에 따라 평가를 수행하라.

---

🔹 단계 1: 질문을 핵심 요소로 분해하라.  
예: 질문 = "A와 B에 대해 설명해달라" → 요소1: A에 대한 설명, 요소2: B에 대한 설명

🔹 단계 2: 각 요소가 답변에서 명시적으로 언급되었는지 여부를 판단하라.  
답변에 요소가 간접적으로 암시되었더라도 명시적으로 언급되지 않으면 "없음"으로 판단한다.

🔹 단계 3: 아래 평가 기준에 따라 “있다 / 없다”로 결과를 출력한다.  
특히 1번 항목(질문 이해 및 연관성)은, 질문의 모든 구성 요소가 빠짐없이 포함된 경우에만 "있다"로 평가한다.  
하나라도 누락되었으면 반드시 "없다"로 평가한다.

---

### 면접 질문:
{question}

### 지원자 답변:
{answer}


### 질문 구성 요소:
- 요소1: ...
- 요소2: ...
(예시: Transformer의 장점 설명, 프로젝트 활용 경험 설명)


### 답변 대응 여부:
- 요소1: 있음 / 없음
- 요소2: 있음 / 없음
※ 절대 기준: 질문의 구성 요소 중 하나라도 빠지면 "질문 이해 및 연관성"은 "없다"로 평가할 것.
---
출력 예시
---                                       
### 평가 결과:
[인성 평가]  
1. 질문 이해 및 연관성: [ ]  
2. 자기 성찰 및 경험 활용: [ ]  
3. 태도 및 소통 역량: [ ]

[기술 평가]  
4. 지식의 정확성과 깊이: [ ]  
5. 적용 및 실무 경험: [ ]  
6. 문제 해결 및 응용력: [ ]



""")



In [61]:
def evaluate_answer(question, answer, model_name="gpt-4o"):
    llm = ChatOpenAI(model=model_name, temperature=0)
    chain = LLMChain(llm=llm, prompt=prompt_template)
    result = chain.run({
        "question": question,
        "answer": answer
    })
    return result


In [64]:
question = "회로에서 인덕터와 커패시터의 역할과 차이점에 대해 설명하고, 실제 회로 설계 시 두 부품을 어떻게 활용하는지 예를 들어 설명해주세요."

answer = (
    "인덕터는 자기장을 생성하여 전류의 변화를 저항하는 성질을 가지고 있으며, 커패시터는 전하를 저장하여 전압의 변화를 저항하는 역할을 합니다."
)


In [65]:
result = evaluate_answer(question, answer)
print(result)


### 단계 1: 질문을 핵심 요소로 분해하라.

질문 = "회로에서 인덕터와 커패시터의 역할과 차이점에 대해 설명하고, 실제 회로 설계 시 두 부품을 어떻게 활용하는지 예를 들어 설명해주세요."

- 요소1: 인덕터의 역할 설명
- 요소2: 커패시터의 역할 설명
- 요소3: 인덕터와 커패시터의 차이점 설명
- 요소4: 실제 회로 설계 시 인덕터 활용 예시
- 요소5: 실제 회로 설계 시 커패시터 활용 예시

### 단계 2: 각 요소가 답변에서 명시적으로 언급되었는지 여부를 판단하라.

- 요소1: 있음 (인덕터는 자기장을 생성하여 전류의 변화를 저항하는 성질을 가지고 있음)
- 요소2: 있음 (커패시터는 전하를 저장하여 전압의 변화를 저항하는 역할을 함)
- 요소3: 있음 (인덕터는 전류의 변화를 저항하고, 커패시터는 전압의 변화를 저항함)
- 요소4: 없음 (실제 회로 설계 시 인덕터 활용 예시는 없음)
- 요소5: 없음 (실제 회로 설계 시 커패시터 활용 예시는 없음)

### 평가 결과:

[인성 평가]  
1. 질문 이해 및 연관성: [없다]  
   - 이유: 질문의 모든 구성 요소가 빠짐없이 포함되지 않았음 (요소4, 요소5 누락)
2. 자기 성찰 및 경험 활용: [없다]  
   - 이유: 실제 회로 설계 시 활용 예시가 포함되지 않았음
3. 태도 및 소통 역량: [없다]  
   - 이유: 질문의 요구 사항을 완전히 충족하지 못함

[기술 평가]  
4. 지식의 정확성과 깊이: [있다]  
   - 이유: 인덕터와 커패시터의 역할과 차이점에 대한 설명이 정확함
5. 적용 및 실무 경험: [없다]  
   - 이유: 실제 회로 설계 시 활용 예시가 포함되지 않았음
6. 문제 해결 및 응용력: [없다]  
   - 이유: 실무 적용 예시가 부족하여 응용력 평가가 어려움
